In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *

In [2]:
data = pd.read_csv("../../resources/datasets/split_datasets/CLIP_X_test.csv", index_col=0)
data1 = torch.tensor(data.iloc[0:1].values, dtype=torch.float32)

rider_data = pd.read_csv("../../resources/datasets/split_datasets/aero_X_test.csv", index_col=0)
rider_data = rider_data[['upper_leg', 'lower_leg', 'arm_length', 'torso_length', 'neck_and_head_length', 'torso_width']]
rider1 = torch.tensor(rider_data.iloc[0:1].values, dtype=torch.float32)

# rider_full = 

In [3]:
StandardEvaluations: List[EvaluationFunction] = [
    UsabilityEvaluator(),
    AeroEvaluator(),
    ErgonomicsEvaluator(),
    AestheticsEvaluator(mode="Text"),
    StructuralEvaluator(),
    ValidationEvaluator(),
    FrameValidityEvaluator()
]



evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


c:\Users\Lyle\mambaforge\envs\pytorch_clip\lib\site-packages\sklearn\base.py:329: UserWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
condition = {"Rider": rider1, "Use Case": np.array([1, 0, 0]), "Text": "Sporty Bike"}

In [5]:
scores = evaluator(torch.tensor(data.values, dtype=torch.float32), condition)

c:\Users\Lyle\Documents\Files\DeCoDE\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\aero_predictor.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  upper_leg_width = torch.tensor((torso_width/2 - 0.16)/2 + 0.14, device=device)


In [6]:
print(scores)

tensor([[   0.5265,   22.6199,    0.8513,  ...,  -76.6354,  -97.5000,
           -0.5000],
        [   0.2212,    7.7730,   62.0934,  ...,   76.3646, -179.0000,
           -0.5000],
        [   0.4809,   24.8670,    1.8961,  ..., -122.7396, -160.5000,
           -0.5000],
        ...,
        [   0.3368,   22.3032,    4.3629,  ...,  -52.7562, -176.0000,
           -0.5000],
        [   0.3506,   22.3979,    4.3632,  ...,  -58.1107, -176.0000,
           -0.5000],
        [   0.5507,   22.6195,    0.6076,  ...,  -81.2910, -100.0000,
           -0.5000]], grad_fn=<CopySlices>)


In [7]:
objective_scores = scores[:, isobjective]
constraint_scores = scores[:, ~isobjective]
print(objective_scores)
print(constraint_scores)

tensor([[ 0.5265, 22.6199,  0.8513,  ...,  2.0226,  1.2949,  1.3140],
        [ 0.2212,  7.7730, 62.0934,  ...,  0.9821,  2.1526,  2.1006],
        [ 0.4809, 24.8670,  1.8961,  ...,  0.8124,  2.3671,  2.5386],
        ...,
        [ 0.3368, 22.3032,  4.3629,  ...,  0.3775,  0.8025,  1.3432],
        [ 0.3506, 22.3979,  4.3632,  ...,  0.3321,  0.6779,  1.0987],
        [ 0.5507, 22.6195,  0.6076,  ...,  4.5765,  2.3677,  2.6635]],
       grad_fn=<IndexBackward0>)
tensor([[   1.2220,    0.7759, -139.3000,  ...,  -76.6354,  -97.5000,
           -0.5000],
        [   0.9097,    0.9507,  -25.0000,  ...,   76.3646, -179.0000,
           -0.5000],
        [   0.7730,    1.1335, -161.0000,  ..., -122.7396, -160.5000,
           -0.5000],
        ...,
        [   0.8024,    0.8337, -150.0000,  ...,  -52.7562, -176.0000,
           -0.5000],
        [   0.7746,    0.8257, -150.0000,  ...,  -58.1107, -176.0000,
           -0.5000],
        [   1.3636,    1.0577, -150.7000,  ...,  -81.2910, -100.0

In [8]:
# df_evaluator = construct_dataframe_evaluator(StandardEvaluations)

In [9]:
# scores, requirement_types = df_evaluator(data, condition)
# isobjective = np.array(isobjective).astype(np.bool_)

# #index columns using boolean isobjective [True, False, ...]
# objective_scores = scores.iloc[:, isobjective]
# constraint_scores = scores.iloc[:, ~isobjective]

# display(objective_scores)
# display(constraint_scores)

In [10]:
from pymoo.indicators.hv import HV

In [11]:
#replace nan with -inf in objective_scores
objective_scores = scores[:, isobjective].detach().numpy()
constraint_scores = scores[:, ~isobjective].detach().numpy()

In [12]:
validity_mask = np.all(constraint_scores <= 0, axis=1)

In [13]:
constraint_scores.shape

(4512, 14)

In [14]:
(constraint_scores <= 0).any(axis=0)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [15]:
np.max(np.sum(constraint_scores <= 0, axis=1))

13

In [16]:
def sample_riders(num_samples, split = "test"):
    # Sample random riders from the rider data
    if split == "test":
        rider_data = pd.read_csv("../../resources/datasets/split_datasets/aero_X_test.csv", index_col=0)
        rider_data = rider_data[['upper_leg', 'lower_leg', 'arm_length', 'torso_length', 'neck_and_head_length', 'torso_width']]
    elif split == "train":
        rider_data = pd.read_csv("../../resources/datasets/split_datasets/aero_X_train.csv", index_col=0)
        rider_data = rider_data[['upper_leg', 'lower_leg', 'arm_length', 'torso_length', 'neck_and_head_length', 'torso_width']]
    else:
        raise ValueError("Invalid split. Choose 'train' or 'test'.")
    #sample num_samples with replacement
    sampled_riders = rider_data.sample(n=num_samples, replace=True).values
    return sampled_riders

def sample_use_case(num_samples, split=None):    
    # Randomly pick indices 0, 1 or 2
    idx = np.random.choice(3, size=num_samples, replace=True)
    
    # Convert to one-hot
    onehots = np.eye(3, dtype=int)[idx]
    
    return onehots



In [17]:
print(sample_riders(10, split="test"))
print(sample_use_case(10, split="test"))


[[0.36839189 0.53781058 0.6354895  0.55907532 0.28869031 0.34677888]
 [0.38729716 0.49600984 0.65002828 0.44754846 0.2932505  0.31611478]
 [0.40985527 0.50146022 0.63184083 0.51240498 0.30441242 0.31214387]
 [0.36215663 0.56522173 0.69227694 0.5132192  0.28278154 0.3537447 ]
 [0.37125614 0.5745918  0.6153465  0.54175288 0.30891703 0.30913365]
 [0.35505295 0.51426347 0.63120117 0.51664219 0.30446752 0.36496799]
 [0.33686244 0.49016176 0.63911105 0.4804817  0.28954591 0.3932825 ]
 [0.38816035 0.52267487 0.71674263 0.52578984 0.30826399 0.31692819]
 [0.38735228 0.51392435 0.64918266 0.51591803 0.29034212 0.3323071 ]
 [0.38558551 0.52925413 0.70391127 0.50032132 0.27514465 0.30793783]]
[[0 1 0]
 [0 0 1]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [1 0 0]
 [0 0 1]]


In [18]:
import pygmo as pg

def compute_ref_point(ref_scores):
    ref_scores[np.isnan(ref_scores)] = -float("inf")
    ref_point = np.max(ref_scores, axis=0)
    return ref_point

def hypervolume(objective_scores, constraint_scores):
    ref_point = compute_ref_point(objective_scores)

    validity_mask = np.all(constraint_scores <= 0, axis=1)
    valid_objective_scores = objective_scores[validity_mask]
    if valid_objective_scores.size == 0:
        return 0.0
    valid_objective_scores[np.isnan(valid_objective_scores)] = float("inf")
    valid_objective_scores = valid_objective_scores/ref_point
    valid_objective_scores = np.clip(valid_objective_scores, a_min=0, a_max=1)
    scaled_ref_point = np.ones_like(ref_point)

    hv = pg.hypervolume(valid_objective_scores)
    hv_value = hv.compute(ref_point=scaled_ref_point)
    return hv_value

print(hypervolume(objective_scores, np.zeros_like(constraint_scores)))

0.3827758449113655


In [19]:
subset = objective_scores_capped[:1000,:]

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ C:\Users\Lyle\AppData\Local\Temp\ipykernel_26112\2870073357.py:1 in <module>                     │
│                                                                                                  │
│ [Errno 2] No such file or directory:                                                             │
│ 'C:\\Users\\Lyle\\AppData\\Local\\Temp\\ipykernel_26112\\2870073357.py'                          │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'objective_scores_capped' is not defined

In [ ]:
subset.shape

(1000, 10)

In [ ]:
import pygmo as pg
hv = pg.hypervolume(objective_scores_capped/ref_point)
hv.compute(np.ones_like(ref_point))

0.22917476232497253